# NB_lab_test_trends

Builds `<catalog>.gold.lab_test_trends` (one row per test type per calendar month) per the STTM `sttm_silver_to_gold.xlsx`. Driving table `silver.lab_results` grouped by `test_name`, year and month of `test_date`, with an optional calendar join for month/quarter names and a LAG() month-over-month abnormal-rate trend. Full rebuild.

In [0]:
# <catalog> placeholder from the STTM; override via widget / job parameter
dbutils.widgets.text("catalog", "adb_newcaresync")
catalog = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
print(f"Using catalog: {catalog}")

In [0]:
%sql
-- Note: STTM names the calendar source silver.calendar; in this workspace it is silver.dim_calendar
CREATE OR REPLACE TABLE gold.lab_test_trends
COMMENT 'Gold: one row per test type per month with abnormal/critical rates and month-over-month trend (source: silver)'
AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                          AS year,
    month(test_date)                         AS month,
    COUNT(*)                                 AS total_tests,
    COUNT_IF(result_status = 'Abnormal')     AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')     AS critical_test_count
  FROM silver.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    m.*,
    CAST(ROUND(m.abnormal_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(m.critical_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
trend AS (
  SELECT
    r.*,
    LAG(r.abnormal_rate_pct) OVER (PARTITION BY r.test_name ORDER BY r.year, r.month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  t.test_name,
  CAST(t.year AS INT)                                              AS year,
  CAST(t.month AS INT)                                             AS month,
  c.month_name,
  c.quarter_name,
  CAST(t.total_tests AS INT)                                       AS total_tests,
  CAST(t.abnormal_test_count AS INT)                               AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                               AS critical_test_count,
  t.abnormal_rate_pct,
  t.critical_rate_pct,
  CAST(t.prev_month_abnormal_rate_pct AS DECIMAL(5,2))             AS prev_month_abnormal_rate_pct,
  CAST(t.abnormal_rate_pct - t.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM trend t
LEFT JOIN silver.dim_calendar c
  ON c.date_key = CAST(date_format(make_date(t.year, t.month, 1), 'yyyyMMdd') AS INT);

In [0]:
record_count = spark.table("gold.lab_test_trends").count()
dbutils.notebook.exit(str(record_count))